# SUBIT‑Lingua v3.0 — Playground

A fully interactive environment for exploring:
- forms (6‑bit)
- words (12‑bit)
- sequences (n×6‑bit)
- encoding / decoding
- lexicon lookup
- structural visualization

This notebook is self‑contained and uses no external dependencies.

## 1. Core Definitions
SUBIT‑Lingua phonology, bit‑mapping, and axis structure.

In [ ]:
AXES = ["K", "T", "M"]
VOWELS = ["A", "E", "O", "U"]

BITS = {
    "A": "00",
    "E": "01",
    "O": "10",
    "U": "11",
}

INV_BITS = {v: k for k, v in BITS.items()}

## 2. Encoder
Encode forms, words, and sequences into bitstreams.

In [ ]:
import re

FORM_RE = re.compile(r"^[K][AEOU]-[T][AEOU]-[M][AEOU]$")
WORD_RE = re.compile(r"^[K][AEOU]-[T][AEOU]-[M][AEOU]\s+–\s+[K][AEOU]-[T][AEOU]-[M][AEOU]$")

def encode_form(form: str) -> str:
    if not FORM_RE.fullmatch(form):
        raise ValueError(f"Invalid form: {form}")
    syllables = form.split("-")
    vowels = [s[1] for s in syllables]
    return "".join(BITS[v] for v in vowels)

def encode_word(word: str) -> str:
    if not WORD_RE.fullmatch(word):
        raise ValueError(f"Invalid word: {word}")
    left, right = [p.strip() for p in word.split("–")]
    return encode_form(left) + encode_form(right)

def encode_sequence(forms: list[str]) -> str:
    return "".join(encode_form(f) for f in forms)

def encode(obj):
    if isinstance(obj, str):
        if "–" in obj:
            return encode_word(obj)
        return encode_form(obj)
    if isinstance(obj, list):
        return encode_sequence(obj)
    raise TypeError("encode() accepts a form, word, or list of forms.")

## 3. Decoder
Decode 6‑bit, 12‑bit, and n×6‑bit bitstreams back into SUBIT‑Lingua forms.

In [ ]:
FORM_BITS_RE = re.compile(r"^[01]{6}$")
WORD_BITS_RE = re.compile(r"^[01]{12}$")
SEQ_BITS_RE  = re.compile(r"^[01]+$")

def decode_form_bits(bits: str) -> str:
    if not FORM_BITS_RE.fullmatch(bits):
        raise ValueError(f"Invalid 6-bit form: {bits}")
    v1 = INV_BITS[bits[0:2]]
    v2 = INV_BITS[bits[2:4]]
    v3 = INV_BITS[bits[4:6]]
    return f"K{v1}-T{v2}-M{v3}"

def decode_word_bits(bits: str) -> str:
    if not WORD_BITS_RE.fullmatch(bits):
        raise ValueError(f"Invalid 12-bit word: {bits}")
    inner = decode_form_bits(bits[0:6])
    outer = decode_form_bits(bits[6:12])
    return f"{inner} – {outer}"

def decode_sequence_bits(bits: str) -> list[str]:
    if not SEQ_BITS_RE.fullmatch(bits):
        raise ValueError(f"Invalid bitstream: {bits}")
    if len(bits) % 6 != 0:
        raise ValueError("Bitstream length must be a multiple of 6.")
    return [decode_form_bits(bits[i:i+6]) for i in range(0, len(bits), 6)]

def decode(bits: str):
    if len(bits) == 6:
        return decode_form_bits(bits)
    if len(bits) == 12:
        return decode_word_bits(bits)
    return decode_sequence_bits(bits)

## 4. Form Playground
Try encoding and decoding forms.

In [ ]:
form = "KA-TE-MO"
bits = encode(form)
decoded = decode(bits)
form, bits, decoded

## 5. Word Playground
Encode and decode 12‑bit form‑pairs.

In [ ]:
word = "KA-TE-MO – KU-TA-ME"
bits = encode(word)
decoded = decode(bits)
word, bits, decoded

## 6. Sequence Playground
Encode and decode higher‑order forms.

In [ ]:
seq = ["KA-TE-MO", "KU-TA-ME", "KE-TO-MU"]
bits = encode(seq)
decoded = decode(bits)
seq, bits, decoded

## 7. Bitstream Visualizer
Visualize bitstreams in grouped 2‑bit segments.

In [ ]:
def visualize(bits: str):
    return " ".join(bits[i:i+2] for i in range(0, len(bits), 2))

visualize(encode("KA-TE-MO – KU-TA-ME"))

## 8. SUBIT Cube (Text Visualization)
A simple structural visualization of the 64‑form space.

In [ ]:
def form_from_bits(bits):
    return decode_form_bits(bits)

cube = [[form_from_bits(f"{i:06b}") for i in range(r*8, (r+1)*8)] for r in range(8)]
cube

## 9. Lexicon Lookup (On‑Demand)
Generate any of the 4096 words without loading the full lexicon.

In [ ]:
def word_from_indices(i, j):
    inner = decode_form_bits(f"{i:06b}")
    outer = decode_form_bits(f"{j:06b}")
    return f"{inner} – {outer}", encode(f"{inner} – {outer}")

word_from_indices(0, 37)